# 第26讲：tokenizer_supply

按原单元编号阅读。配套讲解见本讲的解释.md，运行方法与记录见复现目录。


### 原单元 0（code）

这个配套接口例子解释数字和空格怎样影响编码边界，可与主 notebook 的编码调用对照。


In [1]:
# 这个配套接口例子解释数字和空格怎样影响编码边界，可与主 notebook 的编码调用对照。
# 字节BPE库；后续get_encoding加载词表，不加载语言模型。
import tiktoken

### 原单元 1（code）

同一cl100k编码对象，本次使用完整真实缓存；get_encoding重复调用会复用注册表对象。


In [4]:
# 同一cl100k编码对象，本次使用完整真实缓存；get_encoding重复调用会复用注册表对象。
# 补充实验仍用cl100k，缓存命中返回同对象。
T = tiktoken.get_encoding('cl100k_base')

### 原单元 2（markdown）

数字分块并不等于十进制位独立编码，词表ID也无数学上的数值连续关系。


### 数字的不连续性

### 原单元 3（code）

1234+4567按正则最多三位数字一段，再BPE，得到[4513,19,10,10961,22]。加号本身也是token；不是执行算术。


In [18]:
# 1234+4567按正则最多三位数字一段，再BPE，得到[4513,19,10,10961,22]。加号本身也是token；不是执行算术。
# 补充：确定性编码，空格和数字分块影响ID。
T.encode('1234+4567')

[4513, 19, 10, 10961, 22]

### 原单元 4（code）

遍历原5个ID并逐个decode/print，展示123、4、+、456、7。for逐次赋给t_id，打印五行而不是一个总和。


In [20]:
# 遍历原5个ID并逐个decode/print，展示123、4、+、456、7。for逐次赋给t_id，打印五行而不是一个总和。
# 遍历原显式ID列表，对每个ID单独解码。
for t_id in [4513, 19, 10, 10961, 22]:
    # 显示token字节恢复的文本；本例数字/加号均完整UTF8。
    print(T.decode([t_id]))

123
4
+
456
7


### 原单元 5（markdown）

前导空格可参与token文本，所以world与空格world不一定同ID。


### `world` vs. ` world`

### 原单元 6（code）

无空格world编码14957，不能直接推出所有单词同样行为。


In [5]:
# 无空格world编码14957，不能直接推出所有单词同样行为。
# 补充：确定性编码，空格和数字分块影响ID。
T.encode('world')

[14957]

### 原单元 7（code）

带前导空格world编码1917，空格属于此token的字节内容，不是先统一strip。


In [6]:
# 带前导空格world编码1917，空格属于此token的字节内容，不是先统一strip。
# 补充：确定性编码，空格和数字分块影响ID。
T.encode(' world')

[1917]

### 原单元 8（code）

hello world编码[15339,1917]，第二个token已经含空格，不需再插一个空格ID。


In [9]:
# hello world编码[15339,1917]，第二个token已经含空格，不需再插一个空格ID。
# 补充：确定性编码，空格和数字分块影响ID。
T.encode('hello world')

[15339, 1917]

### 原单元 9（code）

重复编码world仍14957；确定性来自固定词表/正则/输入，不依赖模型采样。


In [15]:
# 重复编码world仍14957；确定性来自固定词表/正则/输入，不依赖模型采样。
# 补充：确定性编码，空格和数字分块影响ID。
T.encode('world')

[14957]

### 原单元 10（code）

解码[1917,14957]得到空格worldworld。decode按token字节拼接，不按词义自动加空格。


In [16]:
# 解码[1917,14957]得到空格worldworld。decode按token字节拼接，不按词义自动加空格。
# 按token字节拼接解码，不按词自动补空格。
T.decode([1917, 14957])

' worldworld'

### 原单元 11（code）

这是留作补充计算的空白代码格。


In [ ]:
# 这是留作补充计算的空白代码格。
